# Performance profiling of distance functions

Anton Antonov    
Maty 2026

---

## Setup

In [19]:
use Math::DistanceFunctions;
use Math::Nearest;

use NativeCall;
use NativeHelpers::Array;

use Data::Summarizers;
use Data::Reshapers;

---

## Nearest neighbors finding

In [35]:
my ($alpha, $m) = sqrt(3)/2, 4;

my $omega = -0.5 + $alpha * 1i;
my @r = (-$m ... $m);
my @pt = do gather {
    for @r -> $a {
        for @r -> $b {
            for @r -> $c {
                for @r -> $d {
                    my $p = $a + $b * 1i + $c * $omega + $d * 1i * $omega;
                    take [$p.re, $p.im]
                }
            }
        }
    }
}

@pt = @pt.map({ copy-to-carray($_, num64) });
my &nf = nearest(@pt, method => 'kdtree', distance-function => &euclidean-distance);

Math::Nearest::Finder(Algorithm::KDimensionalTree(points => 6561, distance-function => &euclidean-distance))

Ad hoc nearest neighbors:

In [36]:
my @distances = cross(^@pt.elems, ^@pt.elems).map({ $_ => euclidean-distance(|@pt[$_]) }).grep({ $_.key.head != $_.key.tail && abs($_.value - 1) ≤ 1e-5 });
#my @nns = @distances».key.classify({ $_.head }).map({ $_.value».tail });
#@nns.elems

[(0 1) => 0.9999999999999998 (0 9) => 0.9999999999999998 (0 81) => 1 (0 82) => 0.9999999999999998 (0 729) => 1 (0 738) => 0.9999999999999998 (1 0) => 0.9999999999999998 (1 2) => 1.0000000000000002 (1 10) => 0.9999999999999998 (1 82) => 1 (1 83) => 1.0000000000000002 (1 730) => 1 (1 739) => 0.9999999999999998 (2 1) => 1.0000000000000002 (2 3) => 0.9999999999999999 (2 11) => 0.9999999999999998 (2 83) => 1 (2 84) => 0.9999999999999999 (2 731) => 1 (2 740) => 0.9999999999999998 (3 2) => 0.9999999999999999 (3 4) => 0.9999999999999999 (3 12) => 0.9999999999999998 (3 84) => 1 (3 85) => 0.9999999999999999 (3 732) => 1 (3 741) => 0.9999999999999998 (4 3) => 0.9999999999999999 (4 5) => 0.9999999999999998 (4 13) => 0.9999999999999998 (4 85) => 1 (4 86) => 0.9999999999999998 (4 733) => 1 (4 742) => 0.9999999999999998 (5 4) => 0.9999999999999998 (5 6) => 0.9999999999999997 (5 14) => 0.9999999999999998 (5 86) => 1 (5 87) => 1.0000000000000002 (5 734) => 0.9999999999999998 (5 743) => 0.99999999999999

In [37]:
&nf(@pt.head, 3)

(NativeCall::Types::CArray[num64].new NativeCall::Types::CArray[num64].new NativeCall::Types::CArray[num64].new)

In [38]:
my @res = do for @pt { &nf($_, [12, 1.1]) };
@res.elems

6561

In [39]:
my @res = do for @pt { &nf($_, 4) };
@res.elems

6561